# DeepSeek + OpenAI SDK: Complete Guide Notebook

This notebook is a full starter for using DeepSeek with the OpenAI Python SDK.

What this notebook now covers:
1. API key and client setup from .env
2. Model smoke tests for all current DeepSeek model names
3. Vision (external URL, inline base64, uploaded file_id)
4. Thinking mode comparison
5. Multi-round conversation flow
6. JSON output mode
7. Tool calls
8. Files API
9. Context caching probe
10. PDF text workflow (extract then ask model)

## Prerequisites

- DEEPSEEK_API_KEY exists in .env
- Kernel is selected and running
- Required packages installed (openai, python-dotenv, pandas, pypdf, requests)

## Important Notes from DeepSeek docs

- OpenAI-compatible base URL: https://api.deepseek.com
- Current model names for chat/responses: deepseek-flash, deepseek-v4-pro
- Vision is supported on deepseek-flash
- Files API supports image uploads (JPEG, PNG, GIF, WebP) for file_id workflows
- For text PDFs, local extraction then model Q&A is used in this notebook

## First Setup
### git setup

```bash
git clone https://github.com/prabh8331/ai-collab-proj.git
git config --global user.email "you@example.com"
git config --global user.name "Your Name"

git checkout -b feature/setup

git push -u origin feature/setup

# git switch -c your-branch-name

git checkout -b feature/test

git push -u origin feature/test

kernal setup

/home/uworker3/Project_venvs_py/ai-agent-mcp/bin/python -m ipykernel install \
  --user \
  --name agents-mcp \
  --display-name "Python (agents-mcp)"
```

## Complete Setup Guide (OpenAI SDK + DeepSeek)

1. Create and activate your Python environment

2. Install dependencies

```bash
pip install openai python-dotenv pandas pypdf requests
```

3. Create .env in the project root

```env
DEEPSEEK_API_KEY=your_real_key_here
```

4. Run notebook cells in order from top to bottom

5. If internet download fails for demo files:
- Save one image at sample_assets/sample_image.jpg
- Save one PDF at sample_assets/sample.pdf

6. You can keep this notebook as a reusable API test bench for future projects

In [11]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
BASE_URL = "https://api.deepseek.com"
MODELS = ["deepseek-flash", "deepseek-v4-pro"]

if not DEEPSEEK_API_KEY:
    raise ValueError("DEEPSEEK_API_KEY is missing. Add it to .env and rerun this cell.")

client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=BASE_URL)

print("DeepSeek client initialized.")
print("Models configured:", MODELS)

DeepSeek client initialized.
Models configured: ['deepseek-flash', 'deepseek-v4-pro']


In [12]:
from time import perf_counter
import pandas as pd

def test_chat_model(model_name: str, question: str = "Reply with exactly: model-ok") -> dict:
    start = perf_counter()
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": question}],
            reasoning_effort="none",
            stream=False,
        )
        elapsed = perf_counter() - start
        text = (response.choices[0].message.content or "").strip()
        usage = getattr(response, "usage", None)
        total_tokens = getattr(usage, "total_tokens", None) if usage else None
        
        return {
            "api": "chat.completions",
            "model": model_name,
            "status": "ok",
            "latency_s": round(elapsed, 2),
            "answer_preview": text[:120],
            "total_tokens": total_tokens,
            "error": "",
        }
    except Exception as exc:
        elapsed = perf_counter() - start
        return {
            "api": "chat.completions",
            "model": model_name,
            "status": "error",
            "latency_s": round(elapsed, 2),
            "answer_preview": "",
            "total_tokens": None,
            "error": str(exc),
        }

In [7]:
chat_results = [test_chat_model(model_name=m) for m in MODELS]
chat_df = pd.DataFrame(chat_results)
display(chat_df)

,api,model,status,latency_s,answer_preview,total_tokens,error
0,chat.completions,deepseek-flash,ok,1.51,model-ok,14,
1,chat.completions,deepseek-v4-pro,ok,0.91,model-ok,14,


In [8]:
def test_responses_model(model_name: str, prompt: str = "Reply with exactly: responses-ok") -> dict:
    start = perf_counter()
    try:
        response = client.responses.create(
            model=model_name,
            instructions="You are a concise assistant.",
            input=prompt,
            reasoning={"effort": "none"},
        )
        elapsed = perf_counter() - start
        text = (response.output_text or "").strip()
        usage = getattr(response, "usage", None)
        total_tokens = getattr(usage, "total_tokens", None) if usage else None
        
        return {
            "api": "responses",
            "model": model_name,
            "status": "ok",
            "latency_s": round(elapsed, 2),
            "answer_preview": text[:120],
            "total_tokens": total_tokens,
            "error": "",
        }
    except Exception as exc:
        elapsed = perf_counter() - start
        return {
            "api": "responses",
            "model": model_name,
            "status": "error",
            "latency_s": round(elapsed, 2),
            "answer_preview": "",
            "total_tokens": None,
            "error": str(exc),
        }

responses_results = [test_responses_model(model_name=m) for m in MODELS]
responses_df = pd.DataFrame(responses_results)
display(responses_df)

,api,model,status,latency_s,answer_preview,total_tokens,error
0,responses,deepseek-flash,ok,0.92,responses-ok,22,
1,responses,deepseek-v4-pro,ok,0.85,responses-ok,21,


## API Guide Coverage and Demos

The next sections contain runnable examples for all requested API guide topics:
- Vision
- Thinking mode
- Multi-round conversation
- JSON output
- Tool calls
- Files API
- Context caching

Notes:
- Vision and file examples use deepseek-flash
- General reasoning examples use deepseek-v4-pro
- Sample files are auto-downloaded when possible; manual fallback paths are printed if download is blocked

In [15]:
from pathlib import Path
import base64
import json
import mimetypes
import random
from pypdf import PdfReader
import requests

ASSETS_DIR = Path("sample_assets")
ASSETS_DIR.mkdir(exist_ok=True)

IMAGE_URL_CANDIDATES = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a9/Example.jpg/640px-Example.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Fronalpstock_big.jpg/640px-Fronalpstock_big.jpg",
]

PDF_URL_CANDIDATES = [
    "https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf",
    "https://www.africau.edu/images/default/sample.pdf",
]

SAMPLE_IMAGE_PATH = None
SAMPLE_IMAGE_SOURCE_URL = None
SAMPLE_PDF_PATH = None

def _download_with_fallback(url_candidates: list[str], output_path: Path, timeout: int = 25) -> dict:
    tried = []
    shuffled = url_candidates[:]
    random.shuffle(shuffled)
    last_error = None

    for url in shuffled:
        tried.append(url)
        try:
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            output_path.write_bytes(response.content)
            return {
                "ok": True,
                "path": str(output_path),
                "source_url": url,
                "bytes": output_path.stat().st_size,
                "tried": tried,
                "error": "",
            }
        except Exception as exc:
            last_error = str(exc)

    return {
        "ok": False,
        "path": str(output_path),
        "source_url": None,
        "bytes": 0,
        "tried": tried,
        "error": last_error or "unknown download error",
    }

def prepare_sample_assets() -> dict:
    global SAMPLE_IMAGE_PATH, SAMPLE_IMAGE_SOURCE_URL, SAMPLE_PDF_PATH

    image_target = ASSETS_DIR / "sample_image.jpg"
    pdf_target = ASSETS_DIR / "sample.pdf"

    image_result = _download_with_fallback(IMAGE_URL_CANDIDATES, image_target)
    pdf_result = _download_with_fallback(PDF_URL_CANDIDATES, pdf_target)

    SAMPLE_IMAGE_PATH = str(image_target) if image_result["ok"] else None
    SAMPLE_IMAGE_SOURCE_URL = image_result["source_url"] if image_result["ok"] else None
    SAMPLE_PDF_PATH = str(pdf_target) if pdf_result["ok"] else None

    guidance = []
    if not image_result["ok"]:
        guidance.append("Place an image at sample_assets/sample_image.jpg")
    if not pdf_result["ok"]:
        guidance.append("Place a PDF at sample_assets/sample.pdf")

    return {
        "image_download": image_result,
        "pdf_download": pdf_result,
        "sample_image_path": SAMPLE_IMAGE_PATH,
        "sample_pdf_path": SAMPLE_PDF_PATH,
        "manual_fallback": guidance,
    }

def upload_image_to_deepseek(image_path: str) -> str:
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    with path.open("rb") as f:
        uploaded = client.files.create(file=f, purpose="user_data")
    return uploaded.id

def analyze_uploaded_image(file_id: str, prompt: str = "Describe this image briefly.") -> str:
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "file", "file_id": file_id},
                ],
            }
        ],
        reasoning_effort="none",
    )
    return (response.choices[0].message.content or "").strip()

def analyze_image_url(image_url: str, prompt: str = "Describe this image briefly.") -> str:
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_url, "detail": "low"}},
                ],
            }
        ],
        reasoning_effort="none",
    )
    return (response.choices[0].message.content or "").strip()

def analyze_inline_base64_image(image_path: str, prompt: str = "Describe this image briefly.") -> str:
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    mime, _ = mimetypes.guess_type(path.name)
    mime = mime or "image/jpeg"

    with path.open("rb") as f:
        image_b64 = base64.b64encode(f.read()).decode("utf-8")

    data_url = f"data:{mime};base64,{image_b64}"

    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": data_url, "detail": "low"}},
                ],
            }
        ],
        reasoning_effort="none",
    )
    return (response.choices[0].message.content or "").strip()

def compare_thinking_modes(model: str = "deepseek-v4-pro", prompt: str = "Give a concise 3-step API debugging plan.") -> pd.DataFrame:
    rows = []
    for effort in ["none", "high"]:
        start = perf_counter()
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            reasoning_effort=effort,
            stream=False,
        )
        elapsed = perf_counter() - start
        message = response.choices[0].message
        reasoning_content = getattr(message, "reasoning_content", None)

        rows.append({
            "model": model,
            "reasoning_effort": effort,
            "latency_s": round(elapsed, 2),
            "answer_preview": (message.content or "")[:140],
            "has_reasoning_content": bool(reasoning_content),
            "total_tokens": getattr(getattr(response, "usage", None), "total_tokens", None),
        })

    return pd.DataFrame(rows)

def run_multi_round_conversation(model: str = "deepseek-v4-pro") -> list[dict]:
    messages = [
        {"role": "system", "content": "You are concise. Keep replies under 35 words."},
    ]

    user_turns = [
        "My project codename is Falcon-42. Reply only with: saved",
        "I am integrating DeepSeek with OpenAI SDK. Give me one best practice.",
        "What was the codename I told you in turn 1?",
    ]

    transcript = []
    for turn in user_turns:
        messages.append({"role": "user", "content": turn})
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            reasoning_effort="none",
        )
        answer = (response.choices[0].message.content or "").strip()
        messages.append({"role": "assistant", "content": answer})
        transcript.append({"user": turn, "assistant": answer})

    return transcript

def json_output_demo(model: str = "deepseek-v4-pro") -> dict:
    response = client.chat.completions.create(
        model=model,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "Return only valid JSON."},
            {
                "role": "user",
                "content": "Provide deployment config with keys: service (string), region (string), replicas (int), features (array of 3 strings).",
            },
        ],
        reasoning_effort="none",
    )

    raw_text = (response.choices[0].message.content or "").strip()
    parsed = json.loads(raw_text)
    return {"raw": raw_text, "parsed": parsed}

def _get_exchange_rate(base_currency: str, target_currency: str) -> dict:
    rates = {
        ("USD", "INR"): 83.0,
        ("USD", "EUR"): 0.92,
        ("EUR", "INR"): 90.2,
        ("INR", "USD"): 0.012,
    }
    base = base_currency.upper()
    target = target_currency.upper()
    rate = rates.get((base, target))
    if rate is None:
        return {
            "ok": False,
            "error": f"Unsupported pair: {base}->{target}",
            "supported_pairs": [f"{a}->{b}" for a, b in rates.keys()],
        }
    return {"ok": True, "base": base, "target": target, "rate": rate}

def run_tool_call_demo(model: str = "deepseek-flash") -> dict:
    tools = [
        {
            "type": "function",
            "function": {
                "name": "get_exchange_rate",
                "description": "Get FX rate for currency conversion.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "base_currency": {"type": "string"},
                        "target_currency": {"type": "string"},
                    },
                    "required": ["base_currency", "target_currency"],
                    "additionalProperties": False,
                },
            },
        }
    ]

    messages = [
        {
            "role": "user",
            "content": "Convert 125 USD to INR using the tool, then explain the multiplication.",
        }
    ]

    first_response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        tool_choice="auto",
        reasoning_effort="none",
    )

    first_message = first_response.choices[0].message
    tool_calls = first_message.tool_calls or []
    if not tool_calls:
        return {
            "status": "no_tool_call",
            "first_reply": (first_message.content or "").strip(),
        }

    call = tool_calls[0]
    args = json.loads(call.function.arguments or "{}")
    tool_result = _get_exchange_rate(args.get("base_currency", ""), args.get("target_currency", ""))

    messages.append(
        {
            "role": "assistant",
            "content": first_message.content or "",
            "tool_calls": [
                {
                    "id": call.id,
                    "type": "function",
                    "function": {
                        "name": call.function.name,
                        "arguments": call.function.arguments,
                    },
                }
            ],
        }
    )
    messages.append(
        {
            "role": "tool",
            "tool_call_id": call.id,
            "content": json.dumps(tool_result),
        }
    )

    second_response = client.chat.completions.create(
        model=model,
        messages=messages,
        reasoning_effort="none",
    )

    final_answer = (second_response.choices[0].message.content or "").strip()
    return {
        "status": "ok",
        "tool_name": call.function.name,
        "tool_args": args,
        "tool_result": tool_result,
        "final_answer": final_answer,
    }

def _cache_hit_tokens(usage_obj) -> int | None:
    if usage_obj is None:
        return None
    details = getattr(usage_obj, "prompt_tokens_details", None)
    if details is not None:
        cached = getattr(details, "cached_tokens", None)
        if cached is not None:
            return cached
    return getattr(usage_obj, "prompt_cache_hit_tokens", None)

def context_cache_probe(model: str = "deepseek-v4-pro", user_id: str = "cache_demo_user") -> dict:
    repeated_context = " ".join(["alpha-beta-gamma-delta" for _ in range(500)])
    prompt = f"Context block: {repeated_context}\n\nReply with exactly: READY"

    request = {
        "model": model,
        "user_id": user_id,
        "messages": [
            {"role": "system", "content": "Return exact requested output."},
            {"role": "user", "content": prompt},
        ],
        "reasoning_effort": "none",
        "temperature": 0,
    }

    first = client.chat.completions.create(**request)
    second = client.chat.completions.create(**request)

    usage_1 = getattr(first, "usage", None)
    usage_2 = getattr(second, "usage", None)

    return {
        "model": model,
        "first_total_tokens": getattr(usage_1, "total_tokens", None) if usage_1 else None,
        "second_total_tokens": getattr(usage_2, "total_tokens", None) if usage_2 else None,
        "first_cache_hit_tokens": _cache_hit_tokens(usage_1),
        "second_cache_hit_tokens": _cache_hit_tokens(usage_2),
        "first_answer": (first.choices[0].message.content or "").strip(),
        "second_answer": (second.choices[0].message.content or "").strip(),
    }

def extract_pdf_text(pdf_path: str, max_chars: int = 12000) -> str:
    path = Path(pdf_path)
    if not path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    reader = PdfReader(str(path))
    chunks = []
    total_chars = 0

    for page in reader.pages:
        text = page.extract_text() or ""
        if text.strip():
            chunks.append(text)
            total_chars += len(text)
        if total_chars >= max_chars:
            break

    combined = "\n\n".join(chunks).strip()
    return combined[:max_chars]

def ask_deepseek_about_pdf(pdf_path: str, question: str, model: str = "deepseek-v4-pro") -> str:
    extracted_text = extract_pdf_text(pdf_path)
    if not extracted_text:
        raise ValueError(
            "No extractable text found in this PDF. If it is scanned, convert pages to images and run vision workflow on those images."
        )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Answer using only the provided document text. If uncertain, say uncertain.",
            },
            {
                "role": "user",
                "content": f"Document:\n{extracted_text}\n\nQuestion: {question}",
            },
        ],
        reasoning_effort="none",
    )
    return (response.choices[0].message.content or "").strip()

In [27]:
IMAGE_URL_CANDIDATES = [
    "https://httpbin.org/image/jpeg",
    "https://raw.githubusercontent.com/github/explore/main/topics/python/python.png",
]

def _guess_image_mime(path: Path) -> str:
    header = path.read_bytes()[:12]
    if header.startswith(b"\x89PNG\r\n\x1a\n"):
        return "image/png"
    if header.startswith(b"GIF87a") or header.startswith(b"GIF89a"):
        return "image/gif"
    if header.startswith(b"RIFF") and header[8:12] == b"WEBP":
        return "image/webp"
    return "image/jpeg"

def analyze_inline_base64_image(image_path: str, prompt: str = "Describe this image briefly.") -> str:
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    mime = _guess_image_mime(path)
    with path.open("rb") as f:
        image_b64 = base64.b64encode(f.read()).decode("utf-8")

    data_url = f"data:{mime};base64,{image_b64}"

    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": data_url, "detail": "low"}},
                ],
            }
        ],
        reasoning_effort="none",
    )
    return (response.choices[0].message.content or "").strip()

## Demo 1: Download sample image and PDF

This cell tries to download demo assets from the internet.
If download is blocked, it prints exactly where to place your own files.

In [17]:
asset_report = prepare_sample_assets()
asset_report

{'image_download': {'ok': True,
  'path': 'sample_assets/sample_image.jpg',
  'source_url': 'https://httpbin.org/image/jpeg',
  'bytes': 35588,
  'tried': ['https://httpbin.org/image/jpeg'],
  'error': ''},
 'pdf_download': {'ok': True,
  'path': 'sample_assets/sample.pdf',
  'source_url': 'https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf',
  'bytes': 13264,
  'tried': ['https://www.africau.edu/images/default/sample.pdf',
   'https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf'],
  'error': ''},
 'sample_image_path': 'sample_assets/sample_image.jpg',
 'sample_pdf_path': 'sample_assets/sample.pdf',
 'manual_fallback': []}

## Demo 2: Vision + Files API

Runs all three image input approaches:
1. External URL
2. Uploaded file_id
3. Inline base64 data URL

In [28]:
if SAMPLE_IMAGE_PATH and Path(SAMPLE_IMAGE_PATH).exists():
    print("Image path:", SAMPLE_IMAGE_PATH)

    if SAMPLE_IMAGE_SOURCE_URL:
        print("\nVision via external URL:")
        print(analyze_image_url(SAMPLE_IMAGE_SOURCE_URL, "List 3 visible elements."))

    print("\nVision via uploaded file_id:")
    uploaded_file_id = upload_image_to_deepseek(SAMPLE_IMAGE_PATH)
    print("uploaded_file_id:", uploaded_file_id)
    print(analyze_uploaded_image(uploaded_file_id, "Describe this image in 2 short bullet points."))

    print("\nVision via inline base64 data URL:")
    print(analyze_inline_base64_image(SAMPLE_IMAGE_PATH, "Describe the scene in one sentence."))
else:
    print("No sample image available.")
    print("Please place an image at sample_assets/sample_image.jpg and rerun this cell.")

Image path: sample_assets/sample_image.jpg

Vision via external URL:
Based on the image, here are 3 visible elements:

1.  **A canine animal** (resembling a jackal or fox) with a grey and tan coat, pointed ears, and a bushy tail.
2.  **The ground** it is walking on, which appears to be dry, dusty, and uneven terrain.
3.  **The animal's shadow**, which is cast on the ground to the left of the animal.

Vision via uploaded file_id:
uploaded_file_id: file-api-e91035ff-de95-4f8c-a7ca-0c302490e57a
- A slender, wolf-like canine with large ears and a distinctive black-tipped tail is walking from right to left across loose, dry ground.
- The animal has a grizzled gray back, reddish-tan legs and flanks, and a white underbelly, with a dark shadow cast to its left.

Vision via inline base64 data URL:
A black-backed jackal walks across a dry, rocky landscape in profile.


## Demo 3: Thinking mode

Compares reasoning_effort none vs high.

In [19]:
thinking_df = compare_thinking_modes(
    model="deepseek-v4-pro",
    prompt="Give me a concise 3-step plan to debug API authentication failures.",
)
display(thinking_df)

,model,reasoning_effort,latency_s,answer_preview,has_reasoning_content,total_tokens
0,deepseek-v4-pro,none,2.88,Here's a concise 3-step plan:\n\n1. **Verify t...,False,172
1,deepseek-v4-pro,high,4.36,1. **Check credentials & request format** \n ...,True,374


## Demo 4: Multi-round conversation

Shows conversation memory by preserving message history across turns.

In [20]:
transcript = run_multi_round_conversation(model="deepseek-v4-pro")
for i, turn in enumerate(transcript, start=1):
    print(f"Turn {i} user: {turn['user']}")
    print(f"Turn {i} assistant: {turn['assistant']}")
    print("-" * 80)

Turn 1 user: My project codename is Falcon-42. Reply only with: saved
Turn 1 assistant: saved
--------------------------------------------------------------------------------
Turn 2 user: I am integrating DeepSeek with OpenAI SDK. Give me one best practice.
Turn 2 assistant: Set a low `temperature` (e.g., 0.2) for consistent, deterministic outputs.
--------------------------------------------------------------------------------
Turn 3 user: What was the codename I told you in turn 1?
Turn 3 assistant: Falcon-42
--------------------------------------------------------------------------------


## Demo 5: JSON output

Forces valid JSON output with response_format json_object.

In [21]:
json_demo = json_output_demo(model="deepseek-v4-pro")
print("Raw JSON text:")
print(json_demo["raw"])

print("Parsed JSON object:")
json_demo["parsed"]

Raw JSON text:
{
  "service": "example-service",
  "region": "us-west-2",
  "replicas": 2,
  "features": ["logging", "monitoring", "caching"]
}
Parsed JSON object:


{'service': 'example-service',
 'region': 'us-west-2',
 'replicas': 2,
 'features': ['logging', 'monitoring', 'caching']}

## Demo 6: Tool calls

Demonstrates function tool calling with a local exchange-rate tool.

In [22]:
tool_demo = run_tool_call_demo(model="deepseek-flash")
tool_demo

{'status': 'ok',
 'tool_name': 'get_exchange_rate',
 'tool_args': {'base_currency': 'USD', 'target_currency': 'INR'},
 'tool_result': {'ok': True, 'base': 'USD', 'target': 'INR', 'rate': 83.0},
 'final_answer': 'Using the retrieved exchange rate of 1 USD = 83.0 INR, the conversion is:\n\n**125 USD × 83.0 INR/USD = 10,375 INR**\n\n### How the multiplication works\n\nThe key idea is **unit cancellation**. The exchange rate is a ratio:\n\n$$\\frac{83.0 \\text{ INR}}{1 \\text{ USD}}$$\n\nWhen you multiply:\n\n$$125 \\text{ USD} \\times \\frac{83.0 \\text{ INR}}{1 \\text{ USD}}$$\n\nThe "USD" appears in the numerator of the first term and the denominator of the rate, so they **cancel out**:\n\n$$= \\frac{125 \\times 83.0 \\text{ INR}}{1} = 10{,}375 \\text{ INR}$$\n\n### The arithmetic step-by-step\n- Break 125 into 100 + 25\n- 100 × 83 = 8,300\n- 25 × 83 = 2,075\n- Sum: 8,300 + 2,075 = **10,375**\n\nSo **125 US dollars equals 10,375 Indian rupees** at this rate.'}

## Demo 7: Context caching

Sends the same large prompt twice and compares cache-hit tokens.

In [24]:
def context_cache_probe(model: str = "deepseek-v4-pro", user_id: str = "cache_demo_user") -> dict:
    repeated_context = " ".join(["alpha-beta-gamma-delta" for _ in range(500)])
    prompt = f"Context block: {repeated_context}\n\nReply with exactly: READY"

    request = {
        "model": model,
        "messages": [
            {"role": "system", "content": "Return exact requested output."},
            {"role": "user", "content": prompt},
        ],
        "reasoning_effort": "none",
        "temperature": 0,
        "extra_body": {"user_id": user_id},
    }

    first = client.chat.completions.create(**request)
    second = client.chat.completions.create(**request)

    usage_1 = getattr(first, "usage", None)
    usage_2 = getattr(second, "usage", None)

    return {
        "model": model,
        "first_total_tokens": getattr(usage_1, "total_tokens", None) if usage_1 else None,
        "second_total_tokens": getattr(usage_2, "total_tokens", None) if usage_2 else None,
        "first_cache_hit_tokens": _cache_hit_tokens(usage_1),
        "second_cache_hit_tokens": _cache_hit_tokens(usage_2),
        "first_answer": (first.choices[0].message.content or "").strip(),
        "second_answer": (second.choices[0].message.content or "").strip(),
    }

In [25]:
cache_report = context_cache_probe(model="deepseek-v4-pro")
cache_report

{'model': 'deepseek-v4-pro',
 'first_total_tokens': 3021,
 'second_total_tokens': 3021,
 'first_cache_hit_tokens': 0,
 'second_cache_hit_tokens': 2944,
 'first_answer': 'READY',
 'second_answer': 'READY'}

## Demo 8: PDF workflow

Extracts text from PDF and asks the model questions about it.

In [26]:
if SAMPLE_PDF_PATH and Path(SAMPLE_PDF_PATH).exists():
    extracted_preview = extract_pdf_text(SAMPLE_PDF_PATH, max_chars=500)
    print("Extracted text preview:")
    print(extracted_preview)
    print("\nModel summary:")
    print(
        ask_deepseek_about_pdf(
            SAMPLE_PDF_PATH,
            "Summarize this document in 5 concise bullet points.",
            model="deepseek-v4-pro",
        )
    )
else:
    print("No sample PDF available.")
    print("Please place a PDF at sample_assets/sample.pdf and rerun this cell.")

Extracted text preview:
Dummy PDF file

Model summary:
Unable to summarize: The provided document text is "Dummy PDF file" and contains no substantive content.
